# Batch Evaluation — Unseen Books
Loads the trained classifiers and evaluates them against books from Project Gutenberg that were **not** used in training.

**Run order:** `BOW.ipynb` → `Book partitioner.ipynb` → `Genre classifier.ipynb` → **this notebook**

**Required files (produced by prior notebooks):**
- `FeatureTrainingData/vectorizer.pkl`
- `model_logistic_regression.pkl`
- `model_svm.pkl`
- `model_random_forest.pkl`
- `model_naive_bayes.pkl`
- `model_xgboost.pkl`
- `model_distilbert/`

Books are downloaded automatically from Project Gutenberg and cached in `TestBooks/`.

In [ ]:
import re
import random
import joblib
import requests
import time
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
)
import matplotlib.pyplot as plt

## Configuration
The `GENRE_BOOKS` dict maps each genre to a list of `(gutenberg_id, title)` pairs.
All books were **not** used in training. Verify or find additional IDs at https://www.gutenberg.org/

In [64]:
PARTITION_SIZE = 400
NUM_PARTITIONS = 200
RANDOM_STATE   = 42
GENRES = ['biography', 'fantasy', 'horror', 'romance', 'sci-fi']

GENRE_BOOKS = {
    'biography': [
        (2376,  'Up from Slavery - Booker T Washington'),
        (2044,  'The Education of Henry Adams'),
        (10378, 'Autobiography - John Stuart Mill'),
        (3913,  'Confessions - Jean-Jacques Rousseau'),
        (4028,  'Autobiography of Benvenuto Cellini'),
        (5140,  'An Autobiography - Anthony Trollope'),
        (11030, 'Incidents in the Life of a Slave Girl'),
        (654,   'Grace Abounding - John Bunyan'),
        (4367,  'Personal Memoirs of Ulysses S Grant'),
        (14214, 'Theodore Roosevelt An Autobiography'),
        (20455, 'Geronimos Story of His Life'),
        (15399, 'The Interesting Narrative - Olaudah Equiano'),
        (16120, 'Narrative of Sojourner Truth'),
        (3296,  'Confessions of Saint Augustine'),
        (14026, 'Struggles and Triumphs - PT Barnum'),
        (9836,  'My Bondage and My Freedom - Frederick Douglass'),
        (4200,  'Diary of Samuel Pepys Vol 1'),
        (1325,  'Twenty Years at Hull-House - Jane Addams'),
        (932,   'Narrative of William Wells Brown'),
        (2805,  'My Life on the Plains - George Custer'),
    ],
    'fantasy': [
        (34,    'The Princess and the Goblin - MacDonald'),
        (225,   'At the Back of the North Wind - MacDonald'),
        (325,   'Phantastes - George MacDonald'),
        (503,   'The Blue Fairy Book - Andrew Lang'),
        (540,   'The Red Fairy Book - Andrew Lang'),
        (574,   'The Well at the Worlds End - William Morris'),
        (2166,  'King Solomons Mines - H Rider Haggard'),
        (3155,  'She - H Rider Haggard'),
        (54,    'The Marvelous Land of Oz - L Frank Baum'),
        (10662, 'The Night Land - WH Hodgson'),
        (765,   'The Moon Pool - A Merritt'),
        (7477,  'The Book of Wonder - Lord Dunsany'),
        (14082, 'The King of Elflands Daughter - Lord Dunsany'),
        (16,    'Peter Pan - JM Barrie'),
        (5,     'Andersens Fairy Tales'),
        (33361, 'Ozma of Oz - L Frank Baum'),
        (150,   'The Violet Fairy Book - Andrew Lang'),
        (1332,  'The Princess and the Curdie - George MacDonald'),
        (5317,  'The Enchanted Island of Yew - L Frank Baum'),
        (151,   'The Yellow Fairy Book - Andrew Lang'),
    ],
    'horror': [
        (174,   'The Picture of Dorian Gray - Oscar Wilde'),
        (389,   'The Great God Pan - Arthur Machen'),
        (8984,  'The House on the Borderland - WH Hodgson'),
        (8486,  'Ghost Stories of an Antiquary - MR James'),
        (4366,  'Can Such Things Be - Ambrose Bierce'),
        (10007, 'Carmilla - Sheridan Le Fanu'),
        (6581,  'The Jewel of Seven Stars - Bram Stoker'),
        (595,   'The Monk - MG Lewis'),
        (3268,  'The Mysteries of Udolpho - Ann Radcliffe'),
        (696,   'The Castle of Otranto - Horace Walpole'),
        (2852,  'The Hound of the Baskervilles - Conan Doyle'),
        (583,   'The Woman in White - Wilkie Collins'),
        (11438, 'The Empty House - Algernon Blackwood'),
        (1155,  'In a Glass Darkly - Sheridan Le Fanu'),
        (5765,  'Uncle Silas - Sheridan Le Fanu'),
        (208,   'The Turn of the Screw - Henry James'),
        (1413,  'The Beetle - Richard Marsh'),
        (9006,  'The Willows - Algernon Blackwood'),
        (175,   'The Phantom of the Opera - Gaston Leroux'),
        (13618, 'The Terror - Arthur Machen'),
    ],
    'romance': [
        (158,   'Emma - Jane Austen'),
        (105,   'Persuasion - Jane Austen'),
        (141,   'Mansfield Park - Jane Austen'),
        (121,   'Northanger Abbey - Jane Austen'),
        (4276,  'North and South - Elizabeth Gaskell'),
        (4274,  'Wives and Daughters - Elizabeth Gaskell'),
        (107,   'Far from the Madding Crowd - Thomas Hardy'),
        (122,   'The Return of the Native - Thomas Hardy'),
        (145,   'Middlemarch - George Eliot'),
        (6688,  'The Mill on the Floss - George Eliot'),
        (599,   'Vanity Fair - WM Thackeray'),
        (2833,  'The Portrait of a Lady - Henry James'),
        (541,   'The Age of Innocence - Edith Wharton'),
        (514,   'Little Women - Louisa May Alcott'),
        (160,   'The Awakening - Kate Chopin'),
        (1023,  'Tess of the dUrbervilles - Thomas Hardy'),
        (153,   'Jude the Obscure - Thomas Hardy'),
        (969,   'The Tenant of Wildfell Hall - Anne Bronte'),
        (2993,  'Agnes Grey - Anne Bronte'),
        (1232,  'Shirley - Charlotte Bronte'),
    ],
    'sci-fi': [
        (5230,  'The Invisible Man - HG Wells'),
        (159,   'The Island of Doctor Moreau - HG Wells'),
        (1013,  'The First Men in the Moon - HG Wells'),
        (18857, 'Journey to the Center of the Earth - Jules Verne'),
        (83,    'From the Earth to the Moon - Jules Verne'),
        (62,    'A Princess of Mars - Edgar Rice Burroughs'),
        (64,    'The Gods of Mars - Edgar Rice Burroughs'),
        (68,    'The Warlord of Mars - Edgar Rice Burroughs'),
        (545,   'At the Earths Core - Edgar Rice Burroughs'),
        (605,   'Pellucidar - Edgar Rice Burroughs'),
        (20869, 'The Skylark of Space - EE Doc Smith'),
        (19141, 'Edisons Conquest of Mars - Garrett Serviss'),
        (624,   'Looking Backward - Edward Bellamy'),
        (10032, 'The Girl in the Golden Atom - Ray Cummings'),
        (201,   'Flatland - Edwin Abbott'),
        (18,    'A Connecticut Yankee in King Arthurs Court'),
        (9741,  'A Honeymoon in Space - George Griffith'),
        (1581,  'When the Sleeper Wakes - HG Wells'),
        (1904,  'In the Days of the Comet - HG Wells'),
        (2488,  'The Mysterious Island - Jules Verne'),
    ],
}

## Load Saved Models

In [ ]:
from xgb_wrapper import XGBStringClassifier      # needed to deserialise model_xgboost.pkl
from bert_wrapper import DistilBertGenreClassifier

vec      = joblib.load('FeatureTrainingData/vectorizer.pkl')
lr       = joblib.load('models/model_logistic_regression.pkl')
svm      = joblib.load('models/model_svm.pkl')
rf       = joblib.load('models/model_random_forest.pkl')
nb       = joblib.load('models/model_naive_bayes.pkl')
xgb_clf  = joblib.load('models/model_xgboost.pkl')
bert_clf = DistilBertGenreClassifier('models/model_distilbert')

print(f'Vectorizer vocabulary size : {len(vec.vocabulary_)} words')
print('Models loaded: Logistic Regression, SVM, Random Forest, Naive Bayes, XGBoost, DistilBERT')

## Download Books from Project Gutenberg
Books are downloaded once and cached in `TestBooks/<genre>/`. Re-running this cell skips already-downloaded files.

Three URL patterns are tried per book to maximise success. A 1-second delay between requests is polite to Gutenberg's servers.

In [ ]:
DOWNLOAD_DIR = Path('UnknownBooks')
DOWNLOAD_DIR.mkdir(exist_ok=True)

HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; book-genre-classifier-research)'}

def _fetch(book_id):
    for url in [
        f'https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt',
        f'https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt',
        f'https://www.gutenberg.org/files/{book_id}/{book_id}.txt',
    ]:
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200:
                return r.content
        except requests.RequestException:
            continue
    return None

downloaded = {genre: [] for genre in GENRES}
failed     = []

print('Downloading books from Project Gutenberg...')
print('Already-downloaded books are skipped.\n')

for genre, books in GENRE_BOOKS.items():
    print(f'--- {genre.upper()} ---')
    genre_dir = DOWNLOAD_DIR / genre
    genre_dir.mkdir(exist_ok=True)
    for book_id, title in books:
        path = genre_dir / f'{book_id}.txt'
        if path.exists():
            print(f'  [cached] {title}')
            downloaded[genre].append((str(path), title))
        else:
            print(f'  Downloading {title}...', end=' ', flush=True)
            content = _fetch(book_id)
            if content:
                path.write_bytes(content)
                downloaded[genre].append((str(path), title))
                print('done')
                time.sleep(1)
            else:
                print('FAILED')
                failed.append((genre, book_id, title))
    print()

total = sum(len(v) for v in downloaded.values())
print(f'Downloaded/cached: {total} books')
if failed:
    print(f'Failed ({len(failed)}): {[t for _,_,t in failed]}')

## Run Predictions
For each downloaded book: strip Gutenberg boilerplate → sample 200 random 100-word partitions → vectorise → majority-vote genre prediction.

In [ ]:
START_PATTERN = re.compile(r'\*{3}\s*START OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}', re.IGNORECASE)
END_PATTERN   = re.compile(r'\*{3}\s*END OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}',   re.IGNORECASE)

def clean_and_partition(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()
    start_match = START_PATTERN.search(raw)
    text = raw[start_match.end():] if start_match else raw
    end_match = END_PATTERN.search(text)
    text = text[:end_match.start()] if end_match else text
    tokens = text.strip().split()
    if len(tokens) < NUM_PARTITIONS + PARTITION_SIZE:
        return None
    random.seed(RANDOM_STATE)
    max_start = len(tokens) - PARTITION_SIZE
    starts = sorted(random.sample(range(max_start), NUM_PARTITIONS))
    return [' '.join(tokens[s : s + PARTITION_SIZE]) for s in starts]

MODEL_PREDICTORS = [('lr', lr), ('svm', svm), ('rf', rf), ('nb', nb), ('xgb', xgb_clf)]

all_preds = []

for genre, books in downloaded.items():
    print(f'Processing {genre} ({len(books)} books)...', end=' ')
    count = 0
    for path, title in books:
        parts = clean_and_partition(path)
        if parts is None:
            print(f'\n  Skipping {title} (too short)')
            continue
        X   = vec.transform(parts)
        row = {'true': genre, 'title': title}
        for key, model in MODEL_PREDICTORS:
            row[key] = Counter(model.predict(X)).most_common(1)[0][0]
        row['bert'] = Counter(bert_clf.predict(parts)).most_common(1)[0][0]
        all_preds.append(row)
        count += 1
    print(f'{count} ok')

print(f'\nTotal books evaluated: {len(all_preds)}')
preds_df = pd.DataFrame(all_preds)
preds_df = preds_df.replace('autobiography', 'biography')

unexpected = (set(preds_df['lr']) | set(preds_df['svm']) | set(preds_df['rf']) |
              set(preds_df['nb']) | set(preds_df['xgb']) | set(preds_df['bert'])) - set(GENRES)
if unexpected:
    print(f'Warning: unexpected labels found: {unexpected}')

## Accuracy Report
Per-genre precision, recall, and F1-score for both classifiers.

In [ ]:
MODEL_COLS = {
    'lr':   'Logistic Regression',
    'svm':  'SVM',
    'rf':   'Random Forest',
    'nb':   'Naive Bayes',
    'xgb':  'XGBoost',
    'bert': 'DistilBERT',
}

print('=== OVERALL ACCURACY ===')
for col, name in MODEL_COLS.items():
    acc = accuracy_score(preds_df['true'], preds_df[col])
    print(f'  {name:25}: {acc:.1%}')
print()

for col, name in MODEL_COLS.items():
    print(f'=== {name.upper()} ===')
    print(classification_report(preds_df['true'], preds_df[col],
                                labels=GENRES, target_names=GENRES, zero_division=0))

## 10-Group Accuracy
Splits the test books into 10 stratified groups (by genre) and computes per-group accuracy for each model.
Mirrors the K-fold CV section in the Genre Classifier to show result variability at the book level.

In [ ]:
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
fold_results = {col: [] for col in MODEL_COLS}

for _, test_idx in kfold.split(preds_df, preds_df['true']):
    fold = preds_df.iloc[test_idx]
    for col in MODEL_COLS:
        fold_results[col].append(accuracy_score(fold['true'], fold[col]))

print(f"{'Model':25}  mean ± std  (10 book-level folds)")
for col, name in MODEL_COLS.items():
    scores = np.array(fold_results[col])
    print(f"  {name:25}  {scores.mean():.4f} ± {scores.std():.4f}")

names  = list(MODEL_COLS.values())
means  = [np.mean(fold_results[c]) for c in MODEL_COLS]
stds   = [np.std(fold_results[c])  for c in MODEL_COLS]

colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(names)))
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(names, means, yerr=stds, color=colors, edgecolor='white', capsize=5)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{m:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.set_title('10-Group Accuracy (mean ± std) — Batch Evaluation Books', fontsize=12)
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('images/batch_eval_kgroup_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: images/batch_eval_kgroup_accuracy.png')

## Confusion Matrices
Seaborn heatmaps for all six models in a 2×3 grid, plus a per-genre grouped accuracy chart.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for ax, (col, name) in zip(axes_flat, MODEL_COLS.items()):
    cm = confusion_matrix(preds_df['true'], preds_df[col], labels=GENRES)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=GENRES, yticklabels=GENRES,
                linewidths=0.5, ax=ax)
    acc = accuracy_score(preds_df['true'], preds_df[col])
    ax.set_title(f'{name}  ({acc:.1%})', fontsize=11, pad=8)
    ax.set_xlabel('Predicted Genre')
    ax.set_ylabel('True Genre')
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('Confusion Matrices — Batch Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/batch_eval_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: images/batch_eval_confusion_matrix.png')

# Per-genre accuracy grouped bar chart
model_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
x, width = np.arange(len(GENRES)), 0.13

fig, ax = plt.subplots(figsize=(13, 4))
for i, (col, name) in enumerate(MODEL_COLS.items()):
    per_genre = preds_df.groupby('true').apply(
        lambda g: (g[col] == g['true']).mean()
    ).reindex(GENRES)
    ax.bar(x + (i - 2.5) * width, per_genre, width,
           label=name, color=model_colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(GENRES)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.15)
ax.set_title('Per-Genre Accuracy — Batch Evaluation')
ax.legend(fontsize=8, ncol=6)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('images/batch_eval_per_genre_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: images/batch_eval_per_genre_accuracy.png')

## Error Analysis
Shows which books were misclassified per model and the genre they were incorrectly assigned to.
Useful for understanding which genres are most confused with each other at the book level.

In [ ]:
print("=== ERROR ANALYSIS — MISCLASSIFIED BOOKS ===\n")
for col, name in MODEL_COLS.items():
    errors = preds_df[preds_df['true'] != preds_df[col]]
    print(f"{'='*60}")
    print(f"  {name} — {len(errors)} / {len(preds_df)} books misclassified")
    print(f"{'='*60}")
    if errors.empty:
        print("  No misclassifications.\n")
        continue
    pair_counts = errors.groupby(['true', col]).size().rename('count')
    print("  Confusion pairs:")
    print(pair_counts.to_string())
    print("\n  Misclassified books:")
    for _, row in errors.iterrows():
        print(f"    [{row['true']}] → [{row[col]}]:  {row['title']}")
    print()

## Per-Book Results
Full breakdown of every book's prediction vs. true genre. Green = correct, red = wrong.

In [ ]:
rename_map = {'title': 'Book', 'true': 'True Genre',
              'lr': 'LR', 'svm': 'SVM', 'rf': 'RF', 'nb': 'NB',
              'xgb': 'XGBoost', 'bert': 'DistilBERT'}
detail_df = preds_df.rename(columns=rename_map).copy()
for col in ['LR', 'SVM', 'RF', 'NB', 'XGBoost', 'DistilBERT']:
    detail_df[f'{col} ✓'] = detail_df[col] == detail_df['True Genre']
cols = ['Book', 'True Genre',
        'LR', 'LR ✓', 'SVM', 'SVM ✓', 'RF', 'RF ✓', 'NB', 'NB ✓',
        'XGBoost', 'XGBoost ✓', 'DistilBERT', 'DistilBERT ✓']
detail_df = detail_df[cols].sort_values(['True Genre', 'Book']).reset_index(drop=True)

check_cols = [c for c in detail_df.columns if c.endswith(' ✓')]
display(
    detail_df.style.apply(
        lambda col: ['background: #d4edda' if v else 'background: #f8d7da' for v in col]
        if col.name in check_cols else ['' for _ in col],
        axis=0,
    )
)